# Session 8 practice: value iteration on the Mars rover

> **Toy example. This notebook does not use the thesis data.** Reinforcement learning needs an environment you can act in, and the EU panel is history, so it is not used here. Why: [notes/rl_and_observational_data.md](../notes/rl_and_observational_data.md).

**Course link:** Machine Learning Specialization, Course 3 Week 3 (states, actions, rewards, return and discount factor γ, policy, the state-action value function Q(s, a), the Bellman equation).

**The course's Mars rover:** six states in a row, numbered 1 to 6. The rover can move left or right. States 1 and 6 are terminal: arriving there ends the episode, with a reward of 100 in state 1 and 40 in state 6. Every other state gives 0. Future rewards are discounted by γ per step.

**Bellman equation:** Q(s, a) = R(s) + γ · max over a′ of Q(s′, a′), where s′ is the state the action leads to. In a terminal state, Q is just its reward. **Value iteration** starts from zeros and applies this equation again and again until nothing changes.

My predictions (fill these in before running)

1. With γ = 0.5, what is Q(4, left)? (The course gives the answer; try to work it out by hand first.)
2. With γ = 0.5, from which states does the best policy go left, and from which right?
3. If γ is lower (0.3, impatient) or higher (0.9, patient), from which states will the policy change direction?

In [1]:
import numpy as np
import pandas as pd

REWARDS = np.array([100, 0, 0, 0, 0, 40], dtype=float)     # states 1..6
TERMINAL = {0, 5}                                           # states 1 and 6 (0-based 0 and 5)
ACTIONS = {"left": -1, "right": +1}

def value_iteration(gamma, tol=1e-12, max_iters=10_000):
    Q = np.zeros((6, 2))
    for it in range(max_iters):
        V = Q.max(axis=1)
        new = np.zeros_like(Q)
        for s in range(6):
            for j, step in enumerate(ACTIONS.values()):
                if s in TERMINAL:
                    new[s, j] = REWARDS[s]                    # the episode ends here
                else:
                    new[s, j] = REWARDS[s] + gamma * V[s + step]   # Bellman equation
        if np.abs(new - Q).max() < tol:
            return new, it + 1
        Q = new
    return Q, max_iters

Q, iters = value_iteration(0.5)
table = pd.DataFrame(Q, index=[f"state {s}" for s in range(1, 7)], columns=list(ACTIONS))
print(f"γ = 0.5: converged after {iters} sweeps")
print(table.round(3).to_string())
print(f"\nCheck: Q(4, left) = {Q[3, 0]:.4g} (the course says 12.5) -> {'OK' if np.isclose(Q[3, 0], 12.5) else 'MISMATCH'}")
course = {(2, 'left'): 50, (2, 'right'): 12.5, (3, 'left'): 25, (3, 'right'): 6.25,
          (4, 'left'): 12.5, (4, 'right'): 10, (5, 'left'): 6.25, (5, 'right'): 20}
ok = all(np.isclose(Q[s - 1, list(ACTIONS).index(a)], v) for (s, a), v in course.items())
print("All eight Q values for states 2–5 match the course's slide:", ok)

γ = 0.5: converged after 6 sweeps
           left   right
state 1  100.00  100.00
state 2   50.00   12.50
state 3   25.00    6.25
state 4   12.50   10.00
state 5    6.25   20.00
state 6   40.00   40.00

Check: Q(4, left) = 12.5 (the course says 12.5) -> OK
All eight Q values for states 2–5 match the course's slide: True


## How the best policy changes with γ

The best action in each state is the one with the larger Q. A small γ makes the rover impatient: a nearer, smaller reward (40, on the right) beats a far, bigger one (100, on the left). A large γ makes it patient.

In [2]:
rows = {}
for gamma in [0.3, 0.5, 0.9]:
    Q, _ = value_iteration(gamma)
    policy = ["(end)" if s in TERMINAL else ("← left" if Q[s, 0] >= Q[s, 1] else "right →") for s in range(6)]
    rows[f"γ = {gamma}: best action"] = policy
    rows[f"γ = {gamma}: best return"] = [f"{v:.2f}" for v in Q.max(axis=1)]
print(pd.DataFrame(rows, index=[f"state {s}" for s in range(1, 7)]).T.to_string())

                     state 1 state 2 state 3  state 4  state 5 state 6
γ = 0.3: best action   (end)  ← left  ← left  right →  right →   (end)
γ = 0.3: best return  100.00   30.00    9.00     3.60    12.00   40.00
γ = 0.5: best action   (end)  ← left  ← left   ← left  right →   (end)
γ = 0.5: best return  100.00   50.00   25.00    12.50    20.00   40.00
γ = 0.9: best action   (end)  ← left  ← left   ← left   ← left   (end)
γ = 0.9: best return  100.00   90.00   81.00    72.90    65.61   40.00


**Reading it:**
- With γ = 0.5 (the course's case), states 2, 3 and 4 go left to the 100, and state 5 goes right to the 40.
- Lowering γ to 0.3 makes state 4 switch to the right as well: the 100 is three steps away and loses too much to discounting.
- Raising γ to 0.9 makes every state go left: waiting a few steps for 100 is worth more than 40 now.

All of this is possible only because the rover's world is fully known and can be replayed as often as needed. That is exactly what an observational panel like the thesis data cannot offer.